<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/02_LiLt_label_projection_and_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate seqeval huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json
import pandas as pd
from pathlib import Path
from huggingface_hub import snapshot_download

In [ ]:
REPO_ID = "Satyamkarn100/FinDoc-AI"
WORK_DIR = Path("./findoc_work").resolve()
HF_DIR = WORK_DIR / "hf_data"
ANN_DIR = WORK_DIR / "fatura_ann"
WORK_DIR.mkdir(parents=True,exist_ok=True)

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=HF_DIR,
    allow_patterns=["manifests/*", "ocr/doctr_default/*.jsonl"],
)

manifest = pd.read_csv(HF_DIR / "manifests/manifest.csv")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
ocr_data = {}
for shard in sorted((HF_DIR / "ocr").rglob("shard_*.jsonl")):
    print("Loading:", shard.name)
    with open(shard, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            ocr_data[rec["filename"]] = rec

print("Total OCR loaded:", len(ocr_data))

Loading: shard_0000.jsonl
Loading: shard_0001.jsonl
Loading: shard_0002.jsonl
Total OCR loaded: 10000


In [ ]:
ORIG_DIR = ANN_DIR / "invoices_dataset_final/Annotations/Original_Format"
if not ORIG_DIR.exists():
    zip_path = WORK_DIR / "FATURA2.zip"
    !wget -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O "{zip_path}"
    !unzip -q -o "{zip_path}" "invoices_dataset_final/Annotations/Original_Format/*" -d "{ANN_DIR}"

print("Manifest rows :", len(manifest))
print("OCR loaded    :", len(ocr_data))
print("JSONs loaded  :", len(list(ORIG_DIR.glob("*.json"))))
print("\nSplit counts:\n", manifest["split"].value_counts())

--2026-11-23 19:06:51--  https://zenodo.org/records/10371464/files/FATURA2.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.40.132, 188.185.42.113, 188.185.45.63, ...
Connecting to zenodo.org (zenodo.org)|188.185.40.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 690717658 (659M) [application/octet-stream]
Saving to: ‘/kaggle/working/findoc_work/FATURA2.zip’

/kaggle/working/fin 100%[===================>] 658.72M  4.86MB/s    in 2m 22s  

2026-11-23 19:09:13 (4.64 MB/s) - ‘/kaggle/working/findoc_work/FATURA2.zip’ saved [690717658/690717658]

Manifest rows : 10000
OCR loaded    : 10000
JSONs loaded  : 10000

Split counts:
 split
train    6400
test     2000
val      1600
Name: count, dtype: int64


In [ ]:
file_path = ORIG_DIR / "Template1_Instance0.json"

with open(file_path,'r') as f:
    ann = json.load(f)

print(ann["TITLE"])

{'bbox': [[328.0, 802.1638], [443.02, 820.1638]], 'text': 'TAX INVOICE'}


In [ ]:
bbox = ann["TITLE"]["bbox"]
x1,y1 = bbox[0]
x2,y2 = bbox[1]

rec = ocr_data['Template1_Instance0.jpg']
w = rec['width']
h = rec['height']


nx1 = min(x1,x2) / w
nx2 = max(x1,x2) / w

ny1 = (h-max(y1,y2)) / h
ny2 = (h-min(y1,y2)) / h

data_box = [nx1,ny1,nx2,ny2]
print(data_box)

[0.5512605042016807, 0.024775505350772845, 0.7445714285714286, 0.04617859690844229]


In [ ]:
gx1, gy1, gx2, gy2 = data_box

for wd in rec["words"]:
    text = wd[0]
    wx1, wy1, wx2, wy2 = wd[2:6]

    ix1 = max(wx1, gx1)
    iy1 = max(wy1, gy1)
    ix2 = min(wx2, gx2)
    iy2 = min(wy2, gy2)

    if ix2 > ix1 and iy2 > iy1:
        inter_area = (ix2 - ix1) * (iy2 - iy1)
        word_area = (wx2 - wx1) * (wy2 - wy1)
        ratio = inter_area / word_area

        if ratio >= 0.5:
            print(text, round(ratio, 2))

TAX 0.97
INVOICE 1.0


In [ ]:
print(" ".join(wd[0] for wd in ocr_data["Template10_Instance0.jpg"]["words"]))

Address:4894 Melanie Pass Apt. 742 Tammyland, SD 42587 US Date: 03-Jan-1994 INVOICE # INV/30-14/832 Due Date : 06-Feb-2007 ITEMS Prove amount sell, Assume against. Terms and Conditions HGH Bill to:Michael Sparks 4894 Melanie Pass Apt. 742 Tammyland, SD 42587 US Tel:*/311)738-8211 Mclean-Cochran Email:cjames@example.com Site:https:/wwww.chase-allen.orgl QUANTITY 1.00 6.00 PRICE $64.54 $62.36 TAX $0.00 $0.00 DISCOUNT(2.14%): (-) 9.39 TOTAL : 441.14 EUR Email:cjames@example.com AMOUNT $64.54 $374.16 will be charged if payment is not made within the due date. Thank you for your business!


# Finalizing the Feilds

In [ ]:
FIELDS = (
    "NUMBER",
    "DATE",
    "DUE_DATE",
    "PO_NUMBER",
    "CUSTOMER",
    "SELLER_NAME",
    "SELLER_ADDRESS",
    "SELLER_EMAIL",
    "SUB_TOTAL",
    "DISCOUNT",
    "TAX",
    "TOTAL",
    "AMOUNT_DUE",
)

In [ ]:
labels = ["0"]
for field in FIELDS:
    labels.append(f"B-{field}")
    labels.append(f"I-{field}")

# Label Encoding
label2id = {}
id2label = {}

for idx,lbl in enumerate(labels):
    label2id[lbl] = idx
    id2label[idx] =lbl

print("Total fields:", len(FIELDS))
print("Total BIO classes:", len(labels))
print("Sample mapping:", list(label2id.items())[:7])


Total fields: 13
Total BIO classes: 27
Sample mapping: [('0', 0), ('B-NUMBER', 1), ('I-NUMBER', 2), ('B-DATE', 3), ('I-DATE', 4), ('B-DUE_DATE', 5), ('I-DUE_DATE', 6)]


In [ ]:
import json, re
from pathlib import Path
from collections import Counter, defaultdict

key_docs = Counter()
key_templates = defaultdict(set)
key_samples = defaultdict(dict)  # har key ke 3 alag templates ke samples

for p in sorted(Path(ORIG_DIR).glob("*.json")):
    m = re.match(r"Template(\d+)_", p.name)
    t = int(m.group(1)) if m else -1
    ann = json.load(open(p, encoding="utf-8"))

    for k, v in ann.items():
        key_docs[k] += 1
        key_templates[k].add(t)
        if len(key_samples[k]) < 3 and t not in key_samples[k]:
            if isinstance(v, dict):
                txt = str(v.get("text", "<no text>"))
            elif isinstance(v, list):
                txt = f"<list len={len(v)}>"
            else:
                txt = f"<{type(v).__name__}>"
            key_samples[k][t] = txt.replace("\n", " | ")[:75]

for k, n in key_docs.most_common():
    print(f"\n[{k}] -> {n} docs | {len(key_templates[k])} templates")
    for t, txt in key_samples[k].items():
        print(f"   T{t:02d}: {txt}")


[TABLE] -> 10000 docs | 50 templates
   T10: <list len=1>
   T11: <list len=1>
   T12: <list len=1>

[INVOICE_INFO] -> 10000 docs | 50 templates
   T10: <list len=0>
   T11: <list len=0>
   T12: <list len=0>

[OTHER] -> 10000 docs | 50 templates
   T10: Address:4894 Melanie Pass Apt. 742 | Tammyland, SD 42587 US |  | Date: 03-J
   T11: TAX INVOICE |  | Invoice number INV/72-06/714 | Invoice Date: 20-Apr-2010 |
   T12: INVOICE |  | Invoice number 2970-559 | Invoice Date: 23-Jan-2002 | Due Date

[LOGO] -> 10000 docs | 50 templates
   T10: <no text>
   T11: <no text>
   T12: <no text>

[DATE] -> 9800 docs | 49 templates
   T10: Date: 03-Jan-1994
   T11: Invoice Date: 20-Apr-2010
   T12: Invoice Date: 23-Jan-2002

[NUMBER] -> 8800 docs | 44 templates
   T10: INVOICE # INV/30-14/832
   T11: Invoice number INV/72-06/714
   T12: Invoice number 2970-559

[SELLER_ADDRESS] -> 8200 docs | 41 templates
   T10: Address:4894 Melanie Pass Apt. 742 | Tammyland, SD 42587 US
   T12: Address:6479 Smith 

In [ ]:
def cannon(ann):
    out = {}
    for k,v in ann.items():
        if k in ("BUYER","BILL_TO"):
            key = "CUSTOMER"

        elif k == "SEND_TO":
            key = "SHIP_TO"


        elif k == "TAX" or k == "GST" or k.startswith("GST("):
            key = "TAX"

        else:
            key = k

        if key not in out:
            out[key] = [v] # {"CUSTOMER": [Joyce_data]}
        else:
            out[key].append(v)

    return out



In [ ]:
sample = json.load(open(Path(ORIG_DIR) / "Template49_Instance167.json", encoding="utf-8"))
res = cannon(sample)
print("Keys:", list(res.keys()))
print("CUSTOMER count:", len(res.get("CUSTOMER", [])))

Keys: ['TABLE', 'INVOICE_INFO', 'CUSTOMER', 'DATE', 'NUMBER', 'SELLER_ADDRESS', 'SELLER_NAME', 'SUB_TOTAL', 'TOTAL', 'TOTAL_WORDS', 'GSTIN', 'OTHER', 'LOGO']
CUSTOMER count: 2
